# Export quantized audio sample files

Collect test clips once, then for each `.tflite` in `models/` write `audio_samples/<stem>.rs` (raw int8 audio for the Rust runners, log-mel int8 for `cnn_mel_tf`). Also writes `src/sample_input.[ch]` from clip 1 for the Zephyr/Axon firmware.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from utils import (
    SAMPLE_RATE,
    TARGET_AUDIO_LEN_TIME,
    TARGET_AUDIO_LEN_MEL,
    DATASET_ROOT,
    REPO_ROOT,
)
from rust_export import (
    collect_test_clips_for_rs,
    make_time_feature_fn,
    make_mel_feature_fn,
    write_audio_sample_quantized_rs,
    write_audio_sample_rs,
    write_sample_input_c,
    write_sample_input_raw_c,
)

MODELS_DIR = REPO_ROOT / "models"
OUT_DIR = REPO_ROOT / "audio_samples"
SRC_DIR = REPO_ROOT / "src"
OUT_NO_QUANTIZED_RS = OUT_DIR / "no_quantized.rs"

In [ ]:
clips = collect_test_clips_for_rs(
    DATASET_ROOT / "testing",
    sample_rate=SAMPLE_RATE,
    target_len=TARGET_AUDIO_LEN_TIME,
    num_per_label=2,
)

write_audio_sample_rs(
    OUT_NO_QUANTIZED_RS,
    clips,
    SAMPLE_RATE,
    generator_name="building/export_audio_sample_rs.ipynb",
)

time_fn = make_time_feature_fn(TARGET_AUDIO_LEN_TIME)
mel_fn = make_mel_feature_fn(TARGET_AUDIO_LEN_MEL)

def feature_fn_for(model_path):
    return mel_fn if model_path.stem == "cnn_mel_tf" else time_fn

tflite_models = sorted(MODELS_DIR.glob("*.tflite"))
if not tflite_models:
    raise RuntimeError(f"No .tflite models in {MODELS_DIR}")

written = []
for model_path in tflite_models:
    out_rs = OUT_DIR / f"{model_path.stem}.rs"
    write_audio_sample_quantized_rs(
        out_rs,
        clips,
        SAMPLE_RATE,
        model_path,
        feature_fn=feature_fn_for(model_path),
        generator_name="building/export_audio_sample_rs.ipynb",
    )
    written.append(out_rs)

cnn_mel = MODELS_DIR / "cnn_mel_tf.tflite"
if cnn_mel.exists():
    write_sample_input_c(
        SRC_DIR,
        clips[0],
        cnn_mel,
        feature_fn=mel_fn,
        generator_name="building/export_audio_sample_rs.ipynb",
    )
    print("wrote", SRC_DIR / "sample_input.c", "and sample_input.h")

    write_sample_input_raw_c(
        SRC_DIR,
        clips[0],
        cnn_mel,
        generator_name="building/export_audio_sample_rs.ipynb",
    )
    print("wrote", SRC_DIR / "sample_input_raw.c")

print("non-quantized:", OUT_NO_QUANTIZED_RS)
print(f"quantized ({len(written)}):")
for p in written:
    print(" -", p)
print("clips=", len(clips), "samples_per_clip=", len(clips[0][1]))